# 🏠 Домашнее задание №1 — Решаем математику

## Цель

Выбить 100\100 по датасету

## Описание

Вам дан датасет с математическими вопросами (уравнения, арифметика, сравнения и др.).  
Каждый вопрос имеет:
- `question_id` — уникальный идентификатор
- `question` — текст задачи  
- `answer` — правильный ответ


**Ваша задача:**
1. Решить задачи с помощью LLM
2. Написать LLM flow, который для каждого вопроса:
   - отправляет вопрос модели
   - получаете результат
Решение может быть с помощью SGR, function_call. Решайте как считаете правильным. Главное результат

4. Прогнать на всём датасете и получить итоговый **score**

## Критерии оценки

- **Accuracy** — доля правильных ответов (совпадение значения)

---

> 💡 Вы можете использовать **любого провайдера** и **любую модель** с поддержкой 

---

## 📊 Оценка ДЗ (10-балльная шкала)

| Критерий | Баллы |
|---|---|
| **Промпт** (применены минимум 2 техники из лекции) | **1** |
| **Accuracy** ниже 85% | **0** |
| **Accuracy** 85–95% | **3** |
| **Accuracy** 95–99% | **4** |
| **Accuracy** 100% | **5** |
| Применение **SGR** и/или **function call** | **+2** |
| Дешёвая модель (< 20 руб / 1M токенов input И output) | **+2** |



#### Примеры дешёвых моделей: `Ministral 3 14B 2512`, `gpt-oss-20b`, `Qwen2.5-VL 7B Instruct`


### ⚠️ Штрафы

| Нарушение | Штраф |
|---|---|
| Модель дороже 200 руб / 1M токенов (input) | **−2** |
| Описание проблемных кейсов в промпте | **−2** |
| Нет retry в случае ошибки при запросе по API | **−1** |


#### "Описание проблемных кейсов в промпте" Это например указать в промпте что для вопроса 'Solve -34*v + 232*v + 52351 = 48985 for v.	' ответ -17

#### За модели средней стоимости (20–200 руб / 1M токенов) штрафов нет.

## 0. Настройка окружения

In [1]:
import warnings

warnings.filterwarnings("ignore")


import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# from src.token_manager import GigaTokenManager
# from src.config import settings
from src.utils import Answer, validate_answers

# token_manager = GigaTokenManager()

## 1. Загрузка датасета

Датасет содержит математические вопросы разных типов. Загрузим его и посмотрим на структуру.

In [2]:
import pandas as pd 
dataset = pd.read_excel(PROJECT_ROOT / "lectures" / "lecture_1" / "hw_1_dataset.xlsx")

In [3]:
dataset

,question_id,question,answer
0,4d637ecc-d8a8-4ef5-9467-259f51ebe692,Solve -282*d + 929 - 178 = -1223 for d.,7
1,27534870-1f8c-41e7-91c8-61a815c00392,Solve 49*l + 45*l - 125 - 63 = 0 for l.,2
2,cb319da5-2d1e-4814-a665-686521c56c02,Solve -64*t + 1387 - 848 + 933 = 0 for t.,23
3,0f9540f4-91cb-4d58-9e91-93931e4a8aeb,Solve 75*g = 192*g - 71*g - 79*g - 264 for g.,-8
4,d05be1f2-eb77-4175-915f-68c3269983ab,Solve -34*v + 232*v + 52351 = 48985 for v.,-17
...,...,...,...
106,d3bbfb3c-1674-4d6c-b883-21e699417455,"Sort 53, 279, 2, 1, 5, 4, -5.","-5, 1, 2, 4, 5, 53, 279"
107,f4c87be8-280e-4f46-90f6-4468641ec58e,"Sort 2, -0.2, -198, -1860.","-1860, -198, -0.2, 2"
108,1585d0d6-0fa0-46b9-9c05-3e91d96994ef,"Sort 5, -175, -4, -7, -30.","-175, -30, -7, -4, 5"
109,36033990-219d-4065-a2a5-08fa1cb8e7e4,"Sort -2, 3936, 15/19 in ascending order.","-2, 15/19, 3936"


In [4]:
print(f"Всего вопросов: {len(dataset)}")

Всего вопросов: 111


## 2. Настройка LLM-клиента

Создайте клиент для работы с LLM. Можно использовать любого провайдера

> ⚠️ Убедитесь, что выбранная модель поддерживает **function calling** (Если будете использовать function calling).

In [ ]:
from openai import OpenAI

# =====================================================================
# TODO: Настройте клиент и модель
# =====================================================================

# client = OpenAI(
#     api_key=token_manager.get_access_token(),
#     base_url="https://gigachat.devices.sberbank.ru/api/v1",
#     http_client=token_manager.http_client,
# )
# model = "GigaChat-2-Max"

client = OpenAI(
    base_url="http://127.0.0.1:8080/v1", api_key="not-needed"
)
model = "ministral-3-8b-int4"



## 3. Обработка одного вопроса

Реализуйте функцию `ask_question()`, которая:

1. Отправляет вопрос модели 
2. Проверяет, вернула ли модель `function_call` (если решаете использовать `function_call`)
3. Если да — парсит аргументы, вызывает нужный function, возвращает результат модели
4. Получает финальный ответ


In [6]:
import re

def extract_final_answer(text: str) -> str:
    # Сначала пробуем убрать <think>...</think>, если модель их использует
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # Затем ищем маркер "Ответ:"
    
    match = re.search(r"Ответ:\s*(.+)", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()

    # Фоллбек — берём последнюю непустую строку
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else text

In [7]:
from typing import List

In [8]:
from typing import Literal
from pydantic import BaseModel

In [9]:
class MathAnswer(BaseModel):
    answer_type: Literal["int", "float", "bool", "list", "str"]
    value: str

In [10]:
from typing import Literal, Optional
from pydantic import BaseModel

class MathAnswer(BaseModel):
    answer_type: Literal["int", "float", "bool", "list", "str"]
    value: Optional[str] = None            # для int / float / bool / str
    list_value: Optional[list[str]] = None # для list — элементы отдельно


SYSTEM_PROMPT = (
    "Ты — математический решатель. Твоя задача — решить следующую математическую задачу, "
    "разбираясь в ней шаг за шагом. "
    "В конце явно укажи финальный ответ в формате: <result>ЗДЕСЬ_ОТВЕТ</result>. "
    "Внутри <result> должен быть только сам ответ, без пояснений и единиц измерения:\n"
    "- если ответ да/нет — пиши True или False (латиницей, с большой буквы, как в Python);\n"
    "- если задача про число в другой системе счисления (base N) — выведи ответ в той же "
    "системе счисления, буквы строчные, без префиксов вроде 0x или подписи основания;\n"
    # "- если ответ — дробь, оставь её в виде a/b, не переводи в десятичную;\n"
    "- если нужно отсортировать/перечислить несколько значений — Сначала обдумай все числа, поразмышляй и перечисли готовый вариант через запятую "
    "в правильном порядке(по заданию), каждое значение в исходном виде (дроби как a/b, а не как десятичная дробь)."
)

SYSTEM_PROMPT_EXTRACT = (
    "Тебе дан текст решения математической задачи с ответом внутри <result>...</result> "
    "(или явно помеченным как 'ответ'/'result', если тегов нет). Извлеки этот ответ, не пересчитывая его.\n\n"
    "Определи answer_type:\n"
    "- 'bool', если ответ True/False;\n"
    "- 'int', если это целое число в десятичной системе;\n"
    "- 'float', если это дробное число в десятичной системе;\n"
    "- 'list', если это несколько значений через запятую (числа и/или дроби);\n"
    "- 'str' во всех остальных случаях (числа в других системах счисления с буквами, "
    "дроби вида a/b как отдельный ответ, и т.п.).\n\n"
    "Если answer_type НЕ 'list': заполни value строкой — точно то значение, что в <result>, "
    "без изменения формата, без лишних пробелов, без обрамляющих символов, "
    "с сохранением регистра (True/False, строчные буквы в других системах счисления).\n\n"
    "Если answer_type == 'list': заполни list_value — список элементов по отдельности, "
    "СТРОГО в том порядке, как они шли в <result>. Не объединяй в одну строку сам, "
    "не добавляй скобки/запятые/пробелы, не меняй формат элементов (дробь a/b оставь дробью)."
)


def ask_question(question: str, attempts: int = 5) -> str | None:
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ],
            temperature=0.2,
        )
        raw_answer = resp.choices[0].message.content
        # print(raw_answer)

        extract_resp = client.chat.completions.parse(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_EXTRACT},
                {"role": "user", "content": f"{raw_answer}"},
            ],
            temperature=0,
            response_format=MathAnswer,
        )

        # mega_resp = client.chat.completions.create(
        #     model = model,
        #     messages = [
        #         {"role": "system", "content": SYSTEM_PROMPT},
        #         {"role": "user", "content": question},
        #         {"role": "assistant", "content" : raw_answer},
        #         {"role": "user", "content": "Почему ответ не верный"}

        #     ]
        # )
        # print(mega_resp.choices[0].message.content)

        import json
        parsed = json.loads(extract_resp.choices[0].message.content)

        if parsed["answer_type"] == "list":
            return ", ".join(parsed["list_value"])
        return parsed["value"]

    except Exception as e:
        print(f"Ошибка: {e}")
        if attempts > 0 :
            print("Повтоная попытка:\n")
            ask_question(question, attempts - 1)
        return None

In [11]:
ask_question("Sort 2, -3, -4, -5, -13309, 3, -140 in decreasing order.")

'3, 2, -3, -4, -5, -140, -13309'

In [24]:
from typing import Literal, Optional
from pydantic import BaseModel
import re

class MathAnswer(BaseModel):
    answer_type: Literal["int", "float", "bool", "list", "str"]
    value: Optional[str] = None
    list_value: Optional[list[str]] = None


SYSTEM_PROMPT = (
    "Ты — математический решатель. Реши задачу, рассуждая шаг за шагом.\n"
    "В конце явно укажи финальный ответ в формате: <result>ЗДЕСЬ_ОТВЕТ</result>.\n"
    "Внутри <result> — только сам ответ, без пояснений и единиц измерения.\n\n"
    "КОНВЕНЦИИ ФОРМУЛИРОВОК (соблюдай буквально):\n"
    "- 'Solve ... for w' — в <result> ТОЛЬКО значение w, даже если ты попутно "
    "нашёл остальные переменные. Никогда не перечисляй их все.\n"
    "- 'What is the difference between A and B?' означает B - A, именно в этом "
    "порядке. 'Subtract A from B' означает B - A.\n"
    "- 'Answer with N decimal places' — округли ровно до N знаков, "
    "хвостовые нули не отбрасывай.\n"
    "- Несколько значений через запятую допустимы ТОЛЬКО если задача просит "
    "отсортировать или перечислить список. Иначе — ровно одно значение.\n\n"
    "ФОРМАТ ЗНАЧЕНИЙ:\n"
    "- да/нет → True или False (латиницей, с большой буквы);\n"
    "- дробь оставляй как a/b, не переводи в десятичную;\n"
    "- число в base N выводи в той же системе, буквы строчные, без 0x "
    "и без подписи основания.\n\n"
    "ПРОЦЕДУРЫ (выполняй явно в рассуждении, не в уме):\n"
    "- base N: выпиши числа в столбик, выровняй по разрядам, считай справа "
    "налево, явно отмечая перенос или заём. Проверь: при сложении результат "
    "длиннее большего слагаемого максимум на один разряд, при вычитании — "
    "не длиннее его. Если не так — пересчитай.\n"
    "- Умножение десятичных: перемножь как целые, затем поставь запятую, "
    "сложив число знаков после запятой у множителей.\n"
    "- Продолжение ряда: выпиши первые разности, затем вторые, затем третьи, "
    "пока не станут постоянными; достраивай обратно снизу вверх.\n"
    "- Сортировка: приведи все числа к сопоставимому виду, сравни попарно, "
    "потом выдай порядок. Помни: -5 больше, чем -140. В <result> верни "
    "значения в ИСХОДНОМ виде.\n"
    "- Системы уравнений: подставь найденное обратно в ОБА уравнения "
    "и проверь равенство, прежде чем писать <result>."
)

SYSTEM_PROMPT_EXTRACT = (
    "Тебе дана исходная ЗАДАЧА и текст РЕШЕНИЯ с ответом внутри <result>...</result> "
    "(или помеченным как 'ответ'/'result', если тегов нет). "
    "Извлеки ответ, НЕ пересчитывая его.\n\n"
    "answer_type:\n"
    "- 'bool' — True/False;\n"
    "- 'int' — целое в десятичной системе;\n"
    "- 'float' — дробное в десятичной системе;\n"
    "- 'list' — ТОЛЬКО если сама ЗАДАЧА просит отсортировать или перечислить "
    "(sort / order / list). Если задача просит одно значение ('Solve ... for w'), "
    "а в решении их несколько — возьми значение именно запрошенной переменной "
    "и НЕ используй 'list';\n"
    "- 'str' — всё остальное (числа в других системах счисления с буквами, "
    "дроби a/b как отдельный ответ).\n\n"
    "Если не 'list': value — точно то значение, что в <result>, без изменения "
    "формата, без лишних пробелов, с сохранением регистра.\n\n"
    "Если 'list': list_value — элементы по отдельности, СТРОГО в порядке из "
    "<result>. Не объединяй в строку, не добавляй скобки, не меняй формат."
)

In [15]:
RE_RESULT = re.compile(r"<result>\s*(.*?)\s*</result>", re.I | re.S)
RE_WANTS_LIST = re.compile(r"\b(sort|order|list|arrange)\b", re.I)
RE_FOR_VAR = re.compile(r"\bfor ([a-z])\b")
RE_DEC_Q = re.compile(r"(\d+)\s+decimal place", re.I)
RE_BASE_Q = re.compile(r"In base (\d+),", re.I)
DIGITS = "0123456789abcdefghijklmnopqrstuvwxyz"


def check_format(question: str, answer: str) -> str | None:
    """Причина отбраковки или None. Ничего не решает — только формат."""
    if not answer:
        return "пустой ответ"

    if "," in answer and not RE_WANTS_LIST.search(question):
        v = RE_FOR_VAR.search(question)
        who = f"переменной '{v.group(1)}'" if v else "одно значение"
        return f"задача просит {who}, а в <result> несколько значений через запятую"

    if RE_WANTS_LIST.search(question) and "," in question and "," not in answer:
        return "задача просит перечислить несколько значений, а в <result> одно"

    m = RE_DEC_Q.search(question)
    if m and not re.fullmatch(rf"-?\d+\.\d{{{m.group(1)}}}", answer):
        return f"нужно число ровно с {m.group(1)} знаками после запятой"

    m = RE_BASE_Q.search(question)
    if m:
        base = int(m.group(1))
        core = answer.lstrip("-").lower()
        bad = [c for c in core if c not in DIGITS[:base]]
        if bad:
            return f"цифры {bad} недопустимы в base {base}"
        if answer != answer.lower():
            return "буквы в числах base N должны быть строчными"
    return None


def ask_question(question: str, attempts: int = 2, verbose: bool = False) -> str | None:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for i in range(attempts + 1):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=0.2 if i == 0 else 0.5,
            )
            raw = resp.choices[0].message.content

            # <result> читаем regex-ом; модель зовём только если тега нет
            found = RE_RESULT.findall(raw or "")
            if found:
                answer = found[-1].strip()
            else:
                ex = client.chat.completions.parse(
                    model=model,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT_EXTRACT},
                        {"role": "user",
                         "content": f"ЗАДАЧА:\n{question}\n\nРЕШЕНИЕ:\n{raw}"},
                    ],
                    temperature=0,
                    response_format=MathAnswer,
                )
                p = ex.choices[0].message.parsed
                answer = ", ".join(p.list_value) if p.answer_type == "list" else p.value

            answer = answer.strip().strip("`'\"")
            if "," in answer:
                answer = ", ".join(x.strip() for x in answer.split(",") if x.strip())

            reason = check_format(question, answer)
            if reason is None:
                return answer

            if verbose:
                print(f"  попытка {i}: {answer!r} — {reason}")
            if i == attempts:
                return answer

            messages += [
                {"role": "assistant", "content": raw},
                {"role": "user", "content":
                    f"Твой ответ не прошёл проверку: {reason}. Перерешай задачу "
                    f"заново по шагам и выведи исправленный <result>."},
            ]

        except Exception as e:
            print(f"Ошибка: {e}")
            if i == attempts:
                return None
    return None

### Проверка на одном вопросе

Прежде чем запускать на всём датасете, проверьте агента на одном примере.

In [18]:
# Тест на одном примере
test_row = dataset.iloc[-6]
print(f"Вопрос:   {test_row['question']}")
print(f"Ожидаемо: {test_row['answer']}")

answer = ask_question(test_row["question"])
print(f"Ответ LLM: {answer}")

if answer is not None:
    print(f"Совпадает: {answer == test_row['answer']}")

Вопрос:   Sort 2, -3, -4, -5, -1339, 3, -140 in decreasing order.
Ожидаемо: 3, 2, -3, -4, -5, -140, -1339
Ответ LLM: 3, 2, -3, -4, -5, -140, -1339
Совпадает: True


## 4. Прогон по всему датасету

Запустите агента на всех вопросах из датасета. Для каждого вопроса:
1. Вызовите `ask_question()` для получения ответа
2. Приведите ответ к нужному типу через `parse_answer()`
3. Сформируйте объект `Answer(question_id=..., answer=...)`

> ⏱️ Этот шаг может занять время — в зависимости от провайдера и количества вопросов.  
> При необходимости добавьте `time.sleep()` между запросами для соблюдения rate limits.

In [19]:
from tqdm.auto import tqdm
import json

answers: list[Answer] = []
failed: list[dict] = []

for idx, row in tqdm(dataset.iterrows(), total=len(dataset), desc="Обработка вопросов"):
    question_id = row["question_id"]
    question = row["question"]

    answer = ask_question(question)

    if answer is None:
        failed.append({"index": idx, "question_id": question_id, "reason": "no_answer"})
        continue

    try:
        answers.append(Answer(question_id=question_id, answer=answer))
    except Exception as e:
        failed.append({
            "index": idx,
            "question_id": question_id,
            "reason": f"parse_error: {e}",
            "answer": answer,
        })

    # Промежуточное сохранение каждые 20 вопросов — не терять прогресс при падении
    if len(answers) % 20 == 0:
        with open("answers_checkpoint.json", "w", encoding="utf-8") as f:
            json.dump([a.model_dump(mode="json") for a in answers], f, ensure_ascii=False, indent=2)

print(f"\nГотово!")
print(f"Ответов получено: {len(answers)} из {len(dataset)}")
print(f"Ошибок: {len(failed)}")

Обработка вопросов: 100%|██████████| 111/111 [18:28<00:00,  9.99s/it]


Готово!
Ответов получено: 111 из 111
Ошибок: 0


## 5. Валидация и подсчёт score

Используем функцию `validate_answers()` из `src.utils` для проверки ответов.  
Валидация проверяет:
- **Значение** — ответ должен **точно совпадать** с эталоном (сравнение по строковому представлению)

### Формула score

$$\text{accuracy} = \frac{\text{правильных ответов}}{\text{всего вопросов в датасете}}$$

In [8]:
import json
from src.utils import Answer

with open("answers_checkpoint.json", "r", encoding="utf-8") as f:
    answers_raw = json.load(f)

answers: list[Answer] = [Answer(**item) for item in answers_raw]

print(f"Загружено ответов: {len(answers)}")

Загружено ответов: 100


In [20]:
# Валидация ответов
result = validate_answers(answers, dataset)

total_questions = len(dataset)
total_answered = len(answers)
total_correct = total_answered - len(result["errors"])

# Score считается от общего числа вопросов в датасете
accuracy = total_correct / total_questions if total_questions > 0 else 0

print("=" * 50)
print(f"📊 РЕЗУЛЬТАТЫ")
print("=" * 50)
print(f"Всего вопросов в датасете: {total_questions}")
print(f"Ответов отправлено:        {total_answered}")
print(f"Не удалось ответить:       {len(failed)}")
print(f"Ошибок валидации:          {len(result['errors'])}")
print(f"Правильных ответов:        {total_correct}")
print(f"")
print(f"🎯 Accuracy: {accuracy:.2%} ({total_correct}/{total_questions})")
print("=" * 50)

📊 РЕЗУЛЬТАТЫ
Всего вопросов в датасете: 111
Ответов отправлено:        111
Не удалось ответить:       0
Ошибок валидации:          13
Правильных ответов:        98

🎯 Accuracy: 88.29% (98/111)


In [21]:
# Детализация ошибок (первые 10)
if result["errors"]:
    print("Примеры ошибок:")
    for err in result["errors"]:
        qid = err["question_id"]
        reason = err["reason"]
        row = dataset[dataset["question_id"] == qid]
        if not row.empty:
            q = row.iloc[0]["question"]
            print(f"\n  [{reason}] {q[:80]}...")
            if reason == "value_mismatch":
                print(f"    Ожидалось: {err['expected']}")
                print(f"    Получено:  {err['actual']}")
else:
    print("✅ Все ответы корректны!")

Примеры ошибок:

  [value_mismatch] Solve -12*t - 4482 = 64*t - 383*t + 141*t for t....
    Ожидалось: 27
    Получено:  -27

  [value_mismatch] Solve -q + 14 = -5*v, -5*v - 44*q + 0*q + 616 = -2*v for v....
    Ожидалось: 0
    Получено:  1232/223

  [value_mismatch] Let o(d) = d**2 - 6*d - 6. Suppose 0 = -23*x + 27*x - 28. Let a be o(x). Let y =...
    Ожидалось: -5
    Получено:  -5/4

  [value_mismatch] What is the next term in -53, -35, 53, 259, 631, 1217, 2065, 3223?...
    Ожидалось: 4739
    Получено:  3691

  [value_mismatch] What is next in 89320, 357315, 803980, 1429321, 2233344, 3216055, 4377460, 57175...
    Ожидалось: 7236376
    Получено:  7236375

  [value_mismatch] What is the difference between -221017 and -1429.06? Answer with 2 decimal place...
    Ожидалось: 219587.94
    Получено:  -219587.94

  [value_mismatch] In base 7, what is -1523 - -45464?...
    Ожидалось: 43641
    Получено:  43636

  [value_mismatch] In base 13, what is -237acb + -4?...
    Ожидалось: -2

In [22]:
# Accuracy по типам ответов
error_qids = {e["question_id"] for e in result["errors"]}
failed_qids = {f["question_id"] for f in failed}

print("Accuracy по типам ответов:")
print("-" * 40)
for atype in ["int", "float", "bool", "list", "str"]:
    subset = dataset[dataset["answer_type"] == atype]
    total = len(subset)
    wrong = sum(1 for _, r in subset.iterrows() 
                if str(r["question_id"]) in error_qids or str(r["question_id"]) in failed_qids)
    correct = total - wrong
    acc = correct / total if total > 0 else 0
    print(f"  {atype:>5}: {acc:.0%}  ({correct}/{total})")

Accuracy по типам ответов:
----------------------------------------


KeyError: 'answer_type'

## 📝 Что нужно сдать

1. Этот ноутбук с **заполненным кодом** (все `TODO` реализованы)
2. Все ячейки должны быть **выполнены** — с выводом результатов
3. В выводе должен быть виден **итоговый score**

---

## 💡 Советы

- **Начните с малого**: сначала отладьте работу на единичных запросах или на малом датасете
- **Промпт имеет значение**: хороший `SYSTEM_PROMPT` сильно влияет на качество ответов
- **Обрабатывайте ошибки**: LLM иногда возвращает неожиданные ответы — ваш код не должен падать
---

GLHF 🚀